# 08 - Future external MIMIC adapter contract

Run this notebook only when the external manifests are ready. The next phase will freeze the Symile ECG-CXR bridge and learn one shared clinical-text adapter from two independent paired datasets:

- MIMIC ECG waveform ↔ machine diagnosis text.
- MIMIC CXR image ↔ JSON radiology report text.

Because the ECG and CXR sensor projectors already share the Symile bridge space, the shared text adapter can connect the two external datasets without pretending they contain patient-level ECG-CXR pairs. External within-modality sensor↔text retrieval must be validated before any cross-dataset analysis.

In [ ]:
from pathlib import Path
import json
import os

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
ECG_JSONL = os.environ.get("MIMIC_ECG_DIAGNOSIS_JSONL", "")
CXR_JSONL = os.environ.get("MIMIC_CXR_REPORT_JSONL", "")
print({"ecg_jsonl": ECG_JSONL or "not configured",
       "cxr_jsonl": CXR_JSONL or "not configured"})


In [ ]:
ECG_REQUIRED = {"sample_id", "waveform_path", "machine_diagnosis", "split"}
CXR_REQUIRED = {"sample_id", "image_path", "report_text", "split"}
templates = {
    "ecg": {"sample_id": "ecg_0001", "waveform_path": "/absolute/path/record",
            "machine_diagnosis": "machine interpretation text", "split": "train"},
    "cxr": {"sample_id": "cxr_0001", "image_path": "/absolute/path/image.jpg",
            "report_text": "radiology report text", "split": "train"},
}
print(json.dumps(templates, indent=2))


In [ ]:
def validate_jsonl(path_value, required, path_key, text_key):
    if not path_value:
        return {"configured": False}
    path = Path(path_value)
    if not path.is_file():
        raise FileNotFoundError(path)
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if not rows:
        raise ValueError(f"Empty JSONL: {path}")
    missing = sorted(required - set(rows[0]))
    if missing:
        raise ValueError(f"{path} is missing keys: {missing}")
    if len({row["sample_id"] for row in rows}) != len(rows):
        raise ValueError(f"Duplicate sample_id values in {path}")
    if any(not str(row[text_key]).strip() for row in rows):
        raise ValueError(f"Blank {text_key} values in {path}")
    missing_files = [row[path_key] for row in rows if not Path(row[path_key]).expanduser().exists()]
    return {"configured": True, "path": str(path), "rows": len(rows),
            "splits": sorted({str(row["split"]) for row in rows}),
            "missing_data_files": len(missing_files),
            "first_missing_files": missing_files[:10]}

report = {
    "ecg": validate_jsonl(ECG_JSONL, ECG_REQUIRED, "waveform_path", "machine_diagnosis"),
    "cxr": validate_jsonl(CXR_JSONL, CXR_REQUIRED, "image_path", "report_text"),
}
print(json.dumps(report, indent=2))
output = ARTIFACT_ROOT / "reports" / "08_external_manifest_validation.json"
output.write_text(json.dumps(report, indent=2))
print("Saved:", output)


## Next implementation step

After the real JSON schemas and paths are confirmed, add a new notebook that freezes HuBERT-ECG, RAD-DINO, and the trained bridge; encodes both external sensors; trains a shared BioClinicalBERT projection using the two paired text sources; and evaluates ECG↔diagnosis and CXR↔report retrieval on patient-disjoint held-out sets. Cross-dataset ECG↔CXR retrieval should be described as semantic retrieval and requires an independently defined relevance standard.